In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch import nn
from datasets import load_dataset, Dataset

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)

from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr




In [ ]:
# =========================================================
# 2. GOOGLE DRIVE
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

MODEL_CHECKPOINT = "distilbert-base-uncased"

LOG_FILE = "/content/drive/MyDrive/DistilBERT_SingleStep_log.csv"

os.makedirs("/content/drive/MyDrive", exist_ok=True)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(["model", MODEL_CHECKPOINT])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 8])
    writer.writerow(["eval_batch_size", 4])
    writer.writerow(["epochs", 10])
    writer.writerow([])



# =========================================================
# 3. SEED SETUP
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




In [ ]:
# =========================================================
# 4. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset from Hugging Face...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())


# =========================================================
# 5. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

NUM_LABELS = len(LABELS)

print("\nLabels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)


# =========================================================
# 6. CONVERT LABELS TO SINGLE-STEP FORMAT
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df


train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())


# =========================================================
# 7. COMBINE AND REDISTRIBUTE DATASET: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_single, val_single, test_single],
    ignore_index=True
)

full_df = full_df[["text"] + LABELS]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n = len(full_df)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_single = full_df[:train_end]
val_single = full_df[train_end:val_end]
test_single = full_df[val_end:]

print("\nRedistributed split sizes:")
print({
    "train": len(train_single),
    "val": len(val_single),
    "test": len(test_single)
})

test_texts = test_single["text"].tolist()


# =========================================================
# 8. CONVERT TO HUGGING FACE DATASET
# =========================================================
train_single = Dataset.from_pandas(
    train_single,
    preserve_index=False
)

val_single = Dataset.from_pandas(
    val_single,
    preserve_index=False
)

test_single = Dataset.from_pandas(
    test_single,
    preserve_index=False
)


# =========================================================
# 9. TOKENIZATION — DISTILBERT TOKENIZER
# =========================================================
tokenizer = DistilBertTokenizer.from_pretrained(
    MODEL_CHECKPOINT
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


train_single = train_single.map(
    tokenize_function,
    batched=True
)

val_single = val_single.map(
    tokenize_function,
    batched=True
)

test_single = test_single.map(
    tokenize_function,
    batched=True
)


# =========================================================
# 10. ADD LABEL VECTOR
# =========================================================
def add_labels(example):
    example["labels"] = [
        float(example[label])
        for label in LABELS
    ]

    return example


train_single = train_single.map(add_labels)
val_single = val_single.map(add_labels)
test_single = test_single.map(add_labels)

train_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)




In [ ]:
# =========================================================
# 11. METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 12. LISTS FOR PLOTTING
# =========================================================
epoch_list = []

train_loss_list = []

val_loss_list = []
val_f1_macro_list = []
val_f1_micro_list = []
val_pearson_mean_list = []

test_loss_list = []
test_f1_macro_list = []
test_f1_micro_list = []
test_pearson_mean_list = []


# =========================================================
# 13. CALLBACK FOR EPOCH LOGGING
# =========================================================
class SaveEpochResultsCallback(TrainerCallback):
    def __init__(self, file_path, test_dataset):
        self.file_path = file_path
        self.test_dataset = test_dataset

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval:
            return

        if metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))
        test_f1_macro = float(test_results.get("test_f1_macro", 0.0))
        test_f1_micro = float(test_results.get("test_f1_micro", 0.0))
        test_pearson_mean = float(test_results.get("test_pearson_mean", 0.0))

        epoch_list.append(epoch)

        train_loss_list.append(train_loss)

        val_loss_list.append(val_loss)
        val_f1_macro_list.append(val_f1_macro)
        val_f1_micro_list.append(val_f1_micro)
        val_pearson_mean_list.append(val_pearson_mean)

        test_loss_list.append(test_loss)
        test_f1_macro_list.append(test_f1_macro)
        test_f1_micro_list.append(test_f1_micro)
        test_pearson_mean_list.append(test_pearson_mean)

        pred = self.trainer_ref.predict(
            self.test_dataset
        )

        logits = pred.predictions
        true_labels = pred.label_ids

        probs = 1 / (1 + np.exp(-logits))
        pred_labels = (probs >= 0.5).astype(int)

        report_dict = classification_report(
            true_labels,
            pred_labels,
            target_names=LABELS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(epoch_list)):
                writer.writerow([
                    epoch_list[i],
                    train_loss_list[i],
                    val_loss_list[i],
                    test_loss_list[i],
                    val_f1_macro_list[i],
                    val_f1_micro_list[i],
                    test_f1_macro_list[i],
                    test_f1_micro_list[i],
                    val_pearson_mean_list[i],
                    test_pearson_mean_list[i]
                ])

            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow([
                "class",
                "precision",
                "recall",
                "f1_score",
                "support"
            ])

            for class_name in LABELS:
                row = report_dict.get(class_name, {})

                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(
            f"\nEpoch {epoch} cumulative losses, final test scores, "
            "and classwise results saved."
        )

        self._inside_eval = False


# =========================================================
# 14. MODEL — DISTILBERT SEQUENCE CLASSIFICATION
# =========================================================
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)


# =========================================================
# 15. TRAINING ARGUMENTS
# =========================================================
training_args = TrainingArguments(
    output_dir="/content/distilbert_output",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=10,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 16. TRAINER
# =========================================================
callback = SaveEpochResultsCallback(
    file_path=LOG_FILE,
    test_dataset=test_single
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_single,
    eval_dataset=val_single,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[callback]
)

callback.trainer_ref = trainer


# =========================================================
# 17. TRAIN
# =========================================================
start = time.time()

trainer.train()

end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print("Epochwise result file saved at:", LOG_FILE)